# Phase 1 — Data Loading & Pipeline
**Munich Bike Counter Data (Munich Cycling Count Stations — Radverkehrszählstellen München)**  
6 stations · 2008–2026 · daily cycling counts + weather

Stations: Arnulf, Erhardt, Hirsch, Kreuther, Margareten, Olympia

## Setup

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, '../src')

RAW_DIR       = Path('../data/raw/rad')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
print('Setup complete.')

Setup complete.


---
## Task 1 — Explore and Load Data
### 1a. What files exist?

In [2]:
all_files = sorted(RAW_DIR.glob('*.csv'))
print(f"Total CSV files: {len(all_files)}\n")

rows = []
for f in all_files:
    size_kb = f.stat().st_size / 1024
    rows.append({'file': f.name[:36], 'size_kb': round(size_kb, 1)})

fdf = pd.DataFrame(rows)
print(fdf.to_string(index=False))

Total CSV files: 43

                                file  size_kb
00c5eaf9-d464-433f-8c9b-ce8a2a16db2b    150.9
052a399a-8d9d-454e-b733-a989bdbe28f7     10.3
05a2178d-3138-4874-a9fd-1ede6f0cedc1    153.4
0a97a624-daa4-4cd8-a820-7d2fa6ffe89a  11063.8
0d5abff4-97c8-4679-be23-72bdbd32cd96      9.2
11f6b146-afac-4d96-bc49-8f49ff545c3c    104.2
190070c2-adec-43b9-bde9-b368d7d4f355    628.1
1b099f15-85fb-4481-b10f-da0e59357099    619.7
205c5c9e-9689-4c28-97cb-e575c6c772ce  11381.9
2f9e99cf-e82d-41fb-990c-67783cf23ab7    150.1
35a5825b-a8c8-4414-92f3-11c880576bd4    146.9
3913d9e6-1be8-4ee4-ab88-1266cbf161f1  11068.1
3ef8aad9-a6b0-4c97-a6b7-8c3a63226b37   9127.0
53ef8c4b-d411-477f-9cf3-b044a4c1aaaa     69.0
561fb0d5-2d27-41bb-bda9-a383d6d42ad1    152.4
6558a5f9-2c96-4e4b-985d-8eb99b7b73b1   7193.1
66be7619-a672-4382-bf88-e3688c5abc2b   5585.8
694b9927-b4d5-4e8f-9c62-09b8ac03c39a  17414.3
7304e087-e02d-4ca1-b4da-5c46b27fa223    149.9
77d30ce3-ecea-4e9f-a30f-bfdf13bc23a9     10.2
784b925b-1d5f

### 1b. Load a sample file — understand structure

In [3]:
# Load a daily-aggregate file (2018): 00c5eaf9...
sample_path = RAW_DIR / '00c5eaf9-d464-433f-8c9b-ce8a2a16db2b.csv'
sample = pd.read_csv(sample_path)
print('Shape:', sample.shape)
print('Columns:', sample.columns.tolist())
print()
sample.head()

Shape: (2190, 14)
Columns: ['_id', 'datum', 'zaehlstelle', 'uhrzeit_start', 'uhrzeit_ende', 'richtung_1', 'richtung_2', 'gesamt', 'min.temp', 'max.temp', 'niederschlag', 'bewoelkung', 'sonnenstunden', 'kommentar']



,_id,datum,zaehlstelle,uhrzeit_start,uhrzeit_ende,richtung_1,richtung_2,gesamt,min.temp,max.temp,niederschlag,bewoelkung,sonnenstunden,kommentar
0,1,2018-01-01,Arnulf,00:00,23:59,148,19,167,3.5,8.4,0.1,75,2.1,NaN
1,2,2018-01-02,Arnulf,00:00,23:59,446,47,493,3.0,7.2,3.2,93,0.5,NaN
2,3,2018-01-03,Arnulf,00:00,23:59,416,38,454,2.9,14.5,16.9,95,0.2,NaN
3,4,2018-01-04,Arnulf,00:00,23:59,353,40,393,4.1,11.8,24.6,96,0.0,NaN
4,5,2018-01-05,Arnulf,00:00,23:59,545,59,604,6.9,12.9,2.7,95,3.4,NaN


In [4]:
# Compare: a 15-minute file (2016) — no weather, 96 rows per day per station
sample_15min = pd.read_csv(RAW_DIR / '3913d9e6-1be8-4ee4-ab88-1266cbf161f1.csv', nrows=10)
print('Columns (15-min file):', sample_15min.columns.tolist())
sample_15min.head()

Columns (15-min file): ['_id', 'datum', 'uhrzeit_start', 'uhrzeit_ende', 'zaehlstelle', 'richtung_1', 'richtung_2', 'gesamt', 'kommentar']


,_id,datum,uhrzeit_start,uhrzeit_ende,zaehlstelle,richtung_1,richtung_2,gesamt,kommentar
0,1,2016-01-01,00:00:00,00:15:00,Arnulf,0,0,0,NaN
1,2,2016-01-01,00:15:00,00:30:00,Arnulf,3,0,3,NaN
2,3,2016-01-01,00:30:00,00:45:00,Arnulf,0,0,0,NaN
3,4,2016-01-01,00:45:00,01:00:00,Arnulf,0,0,0,NaN
4,5,2016-01-01,01:00:00,01:15:00,Arnulf,1,0,1,NaN


### 1c. Column inventory — naming inconsistencies across years

The raw source files use German column names. The pipeline renames all columns to English at load time
(`datum` → `date`, `zaehlstelle` → `station`, `gesamt` → `total`, `richtung_1/2` → `direction_1/2`,
`niederschlag` → `precipitation`, `bewoelkung` → `cloud_cover`, `sonnenstunden` → `sunshine_hours`).

The `min_temp` / `max_temp` columns appear under several raw variants across years:

| Period | `min_temp` variant | `max_temp` variant | Date format | Notes |
|--------|-------------------|--------------------|-------------|-------|
| 2008–2022 | `min.temp` | `max.temp` | `YYYY-MM-DD` | `zaehlstelle` in col 2 |
| 2023–2024 | `min.temp` | `max.temp` | `YYYY.MM.DD` | `zaehlstelle` in col 4 |
| 2025 | `mintemp` | `maxtemp` | `YYYY.MM.DD` | No `kommentar` col |
| 2026 | `min-temp` | `max-temp` | `YYYY.MM.DD` | No `kommentar` col |

In [5]:
# Programmatically confirm all column schemas present in the raw files
schemas = {}
for f in all_files:
    try:
        header = pd.read_csv(f, nrows=0)
        key = tuple(c.lower().strip() for c in header.columns)
        schemas.setdefault(key, []).append(f.name[:8])
    except Exception as e:
        print(f'  Error reading {f.name}: {e}')

print(f'Distinct schemas found: {len(schemas)}\n')
for i, (cols, names) in enumerate(schemas.items(), 1):
    print(f'Schema {i} — {len(names)} file(s)')
    print(f'  Columns  : {list(cols)}')
    print(f'  File IDs : {names}')
    print()

Distinct schemas found: 7

Schema 1 — 15 file(s)
  Columns  : ['_id', 'datum', 'zaehlstelle', 'uhrzeit_start', 'uhrzeit_ende', 'richtung_1', 'richtung_2', 'gesamt', 'min.temp', 'max.temp', 'niederschlag', 'bewoelkung', 'sonnenstunden', 'kommentar']
  File IDs : ['00c5eaf9', '05a2178d', '2f9e99cf', '53ef8c4b', '561fb0d5', '7304e087', '89dbef6c', '8ae44963', '8c752f92', 'b78e7a05', 'cb720004', 'd584bb5e', 'd5dd6fda', 'e281aac1', 'f6d559cc']

Schema 2 — 4 file(s)
  Columns  : ['_id', 'datum', 'uhrzeit_start', 'uhrzeit_ende', 'zaehlstelle', 'richtung_1', 'richtung_2', 'gesamt', 'min-temp', 'max-temp', 'niederschlag', 'bewoelkung', 'sonnenstunden']
  File IDs : ['052a399a', '0d5abff4', '77d30ce3', '93b90485']

Schema 3 — 16 file(s)
  Columns  : ['_id', 'datum', 'uhrzeit_start', 'uhrzeit_ende', 'zaehlstelle', 'richtung_1', 'richtung_2', 'gesamt', 'kommentar']
  File IDs : ['0a97a624', '205c5c9e', '3913d9e6', '3ef8aad9', '6558a5f9', '66be7619', '694b9927', '784b925b', '86962013', '893e1f16', 

---
## Task 2 — Build Merged Dataset

**Strategy:** The raw folder contains two file types per year:
- **Daily aggregate** (`uhrzeit_start` (start time) = 00:00, `uhrzeit_ende` (end time) = 23:59) — one row per (day, station), includes weather  
- **15-minute interval** — 96 rows per (day, station), no weather  

Only the daily files and skip the 15-minute duplicates.

### Column rename mapping

In [6]:
RENAME_MAP = {
    # identifiers
    'datum'        : 'date',
    'zaehlstelle'  : 'station',
    # count columns
    'gesamt'       : 'total',
    'richtung_1'   : 'direction_1',
    'richtung_2'   : 'direction_2',
    # weather columns
    'niederschlag' : 'precipitation',
    'bewoelkung'   : 'cloud_cover',
    'sonnenstunden': 'sunshine_hours',
    # temperature variants
    'min.temp'     : 'min_temp',
    'max.temp'     : 'max_temp',
    'mintemp'      : 'min_temp',
    'maxtemp'      : 'max_temp',
    'min-temp'     : 'min_temp',
    'max-temp'     : 'max_temp',
}

TARGET_COLS = [
    'date', 'station',
    'total', 'direction_1', 'direction_2',
    'min_temp', 'max_temp', 'precipitation', 'cloud_cover', 'sunshine_hours',
]

print('Rename mapping:')
for src, dst in RENAME_MAP.items():
    print(f'  {src:<15} → {dst}')

Rename mapping:
  datum           → date
  zaehlstelle     → station
  gesamt          → total
  richtung_1      → direction_1
  richtung_2      → direction_2
  niederschlag    → precipitation
  bewoelkung      → cloud_cover
  sonnenstunden   → sunshine_hours
  min.temp        → min_temp
  max.temp        → max_temp
  mintemp         → min_temp
  maxtemp         → max_temp
  min-temp        → min_temp
  max-temp        → max_temp


### Load and merge all data

In [7]:
from data_loader import build_master

master = build_master(
    raw_dir     = '../data/raw/rad',
    output_path = '../data/processed/master_bike_data.csv',
)
master.head(10)

Loaded 22 daily files, skipped 21 15-min files.
Master dataset: 35,221 rows saved to ../data/processed/master_bike_data.csv


,date,station,total,direction_1,direction_2,min_temp,max_temp,precipitation,cloud_cover,sunshine_hours
0,2008-06-01,Arnulf,667.0,645.0,22.0,12.5,26.7,0.0,30,13.9
1,2008-06-02,Arnulf,1117.0,1070.0,47.0,15.0,27.9,0.6,44,12.1
2,2008-06-03,Arnulf,1279.0,1240.0,39.0,14.6,21.9,0.2,88,3.2
3,2008-06-04,Arnulf,758.0,715.0,43.0,14.8,21.9,5.1,91,1.4
4,2008-06-05,Arnulf,606.0,569.0,37.0,14.0,20.4,14.2,91,0.6
5,2008-06-06,Arnulf,963.0,919.0,44.0,13.6,19.9,10.0,81,1.7
6,2008-06-07,Arnulf,399.0,390.0,9.0,13.6,18.2,2.0,95,0.7
7,2008-06-08,Arnulf,557.0,542.0,15.0,12.6,22.0,3.9,83,5.4
8,2008-06-09,Arnulf,1244.0,1202.0,42.0,9.5,23.0,0.0,63,11.5
9,2008-06-10,Arnulf,1350.0,1309.0,41.0,12.2,25.7,0.8,48,12.7


In [8]:
print('Columns      :', master.columns.tolist())
print('Shape        :', master.shape)
print('Date range   :', master['date'].min().date(), '→', master['date'].max().date())
print('Stations     :', sorted(master['station'].unique().tolist()))
print('dtypes       :')
print(master.dtypes.to_string())

Columns      : ['date', 'station', 'total', 'direction_1', 'direction_2', 'min_temp', 'max_temp', 'precipitation', 'cloud_cover', 'sunshine_hours']
Shape        : (35221, 10)
Date range   : 2008-06-01 → 2026-04-30
Stations     : ['Arnulf', 'Erhardt', 'Hirsch', 'Kreuther', 'Margareten', 'Olympia']
dtypes       :
date              datetime64[us]
station                      str
total                    float64
direction_1              float64
direction_2              float64
min_temp                 float64
max_temp                 float64
precipitation            float64
cloud_cover                int64
sunshine_hours           float64


---
## Task 3 — Basic Quality Check

In [9]:
from data_loader import get_data_info
get_data_info(master)

Shape          : 35,221 rows × 10 columns
Date range     : 2008-06-01 → 2026-04-30
Stations       : ['Arnulf', 'Erhardt', 'Hirsch', 'Kreuther', 'Margareten', 'Olympia']

Rows per station:
station
Arnulf        6512
Erhardt       5336
Hirsch        6543
Kreuther      5720
Margareten    4567
Olympia       6543

Date range per station:
                first       last
station                         
Arnulf     2008-06-01 2026-04-30
Erhardt    2011-07-01 2026-04-30
Hirsch     2008-06-01 2026-04-30
Kreuther   2008-06-01 2026-04-30
Margareten 2011-07-01 2023-12-31
Olympia    2008-06-01 2026-04-30

Missing values per column:
total          1935
direction_1    1935
direction_2    1935

Negative count values:
  None

Duplicate (date, station) rows: 0


In [10]:
# Missing values broken down per station
miss_by_station = master.groupby('station')['total'].apply(lambda s: s.isna().sum())
miss_by_station = miss_by_station.rename('missing_count_days')
total_by_station = master.groupby('station').size().rename('total_days')
summary = pd.concat([total_by_station, miss_by_station], axis=1)
summary['pct_missing'] = (summary['missing_count_days'] / summary['total_days'] * 100).round(1)
summary

,total_days,missing_count_days,pct_missing
station,,,
Arnulf,6512,412,6.3
Erhardt,5336,24,0.4
Hirsch,6543,554,8.5
Kreuther,5720,309,5.4
Margareten,4567,132,2.9
Olympia,6543,504,7.7


In [11]:
# Weather descriptive stats
weather_cols = ['min_temp', 'max_temp', 'precipitation', 'cloud_cover', 'sunshine_hours']
master[weather_cols].describe().round(2)

,min_temp,max_temp,precipitation,cloud_cover,sunshine_hours
count,35221.00,35221.00,35221.00,35221.00,35221.00
mean,5.70,14.97,2.35,70.25,5.26
std,6.88,9.08,5.37,28.20,4.66
min,-23.10,-9.90,0.00,0.00,0.00
25%,0.50,7.70,0.00,54.00,0.60
50%,5.70,15.10,0.00,79.00,4.30
75%,11.30,22.20,2.20,94.00,9.20
max,21.50,36.90,71.20,100.00,15.70


In [12]:
# Count stats per station (excluding NaN days)
count_cols = ['total', 'direction_1', 'direction_2']
master.groupby('station')[count_cols].describe().round(0)

total                                                         direction_1          ...                   \
             count    mean     std    min     25%     50%     75%      max       count    mean  ...     75%      max   
station                                                                                         ...                    
Arnulf      6100.0  1104.0   721.0    0.0   565.0  1000.0  1489.0   4514.0      6100.0  1046.0  ...  1412.0   4366.0   
Erhardt     5312.0  3853.0  2247.0  102.0  2066.0  3499.0  5434.0  12283.0      5312.0  1964.0  ...  2778.0   6152.0   
Hirsch      5989.0  1033.0   771.0    0.0   417.0   847.0  1500.0   3745.0      5989.0   508.0  ...   739.0   1842.0   
Kreuther    5411.0   397.0   318.0    0.0   158.0   305.0   577.0   1787.0      5411.0   205.0  ...   319.0    996.0   
Margareten  4435.0  2704.0  1446.0    0.0  1596.0  2611.0  3718.0   7652.0      4435.0  1361.0  ...  1868.0   3848.0   
Olympia     6039.0  1738.0  1144.0    0.0   905.0  1517.0  2390.0  15009.0      6039.0   880.0  ...  1197.0  13861.0   

           direction_2                                                         
                 count    mean     std   min     25%     50%     75%      max  
station                                                                        
Arnulf          6100.0    58.0    36.0   0.0    29.0    55.0    84.0    197.0  
Erhardt         5312.0  1888.0  1081.0  36.0  1028.0  1734.0  2653.0   6131.0  
Hirsch          5989.0   525.0   390.0   0.0   214.0   431.0   763.0   1903.0  
Kreuther        5411.0   192.0   139.0   0.0    86.0   165.0   272.0    834.0  
Margareten      4435.0  1343.0   719.0   0.0   790.0  1293.0  1852.0   3804.0  
Olympia         6039.0   858.0   569.0   0.0   439.0   757.0  1202.0  14107.0  

[6 rows x 24 columns]

---
## Task 4 — Save Processed Data

In [13]:
csv_path    = PROCESSED_DIR / 'master_bike_data.csv'
report_path = PROCESSED_DIR / 'data_loading_report.txt'

print(f'master_bike_data.csv  → {csv_path}  ({csv_path.stat().st_size / 1024:.0f} KB)')
if report_path.exists():
    print(f'data_loading_report   → {report_path}  ({report_path.stat().st_size / 1024:.0f} KB)')
else:
    print(f'data_loading_report   → not found (not generated by this pipeline)')
print()
print('First 5 rows of master CSV:')
pd.read_csv(csv_path, nrows=5)

master_bike_data.csv  → ../data/processed/master_bike_data.csv  (1985 KB)
data_loading_report   → not found (not generated by this pipeline)

First 5 rows of master CSV:


,date,station,total,direction_1,direction_2,min_temp,max_temp,precipitation,cloud_cover,sunshine_hours
0,2008-06-01,Arnulf,667.0,645.0,22.0,12.5,26.7,0.0,30,13.9
1,2008-06-02,Arnulf,1117.0,1070.0,47.0,15.0,27.9,0.6,44,12.1
2,2008-06-03,Arnulf,1279.0,1240.0,39.0,14.6,21.9,0.2,88,3.2
3,2008-06-04,Arnulf,758.0,715.0,43.0,14.8,21.9,5.1,91,1.4
4,2008-06-05,Arnulf,606.0,569.0,37.0,14.0,20.4,14.2,91,0.6


---
## Task 5 — Using the data_loader Module

In [14]:
from data_loader import load_master_data, load_station

# Load full dataset
df = load_master_data('../data/processed/master_bike_data.csv')
print('Full dataset:', df.shape)

# Load one station with date filter
arnulf = load_station(
    'Arnulf',
    start_date='2020-01-01',
    end_date='2022-12-31',
    filepath='../data/processed/master_bike_data.csv',
)
print('Arnulf 2020–2022:', arnulf.shape)
arnulf.head()

Full dataset: (35221, 10)
Arnulf 2020–2022: (1096, 10)


,date,station,total,direction_1,direction_2,min_temp,max_temp,precipitation,cloud_cover,sunshine_hours
0,2020-01-01,Arnulf,228.0,201.0,27.0,-2.6,4.9,0.0,21,7.6
1,2020-01-02,Arnulf,708.0,632.0,76.0,-4.6,4.1,0.0,30,7.8
2,2020-01-03,Arnulf,670.0,626.0,44.0,-4.2,8.4,1.9,85,4.2
3,2020-01-04,Arnulf,308.0,283.0,25.0,2.8,7.0,0.8,96,0.0
4,2020-01-05,Arnulf,315.0,277.0,38.0,-1.3,4.7,0.0,74,0.4


---
## Summary

| Item | Value |
|------|-------|
| Raw files processed | 22 of 43 (daily-aggregate only) |
| Rows in master dataset | 35,221 |
| Date coverage | 2008-06-01 → 2026-04-30 |
| Stations | 6 (Arnulf, Erhardt, Hirsch, Kreuther, Margareten, Olympia) |
| Missing count rows | 1,935 (5.5%) — sensor outages |
| Missing weather rows | 0 |
| Duplicate rows | 0 |
| Negative count values | 0 |

**Known issues to handle in Phase 2:**
- Margareten has no data after 2023-12-31
- Erhardt and Margareten start 2011-07-01 (not available for earlier years)
- 1,935 missing count days across all stations (sensor gaps)
- Kreuther has a shorter effective coverage despite starting 2008 (309 missing days)